# Visualizing finite state machines

In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

`dfa2string` and `nfa2string` expose the state encodings and transitions as readable text. `dfa2dot` and `nfa2dot` produce Graphviz DOT source; `dfa2dot` also returns its state-name map. State names are assigned in the deterministic order of F# sets. The initial arrow uses a reserved node name, so integer state `0` cannot collide with it. Labels are escaped before entering DOT.

In [ ]:
let quote (text: string) =
    "\"" + text.Replace("\\", "\\\\").Replace("\"", "\\\"").Replace("\n", "\\n") + "\""

let stateNames states =
    states |> Seq.mapi (fun index state -> state, sprintf "S%d" index) |> Map.ofSeq

The function `dfa2string` converts the given deterministic <span style="font-variant:small-caps;">Fsm</span> into a string.

In [ ]:
let dfa2string (dfa: Dfa<'state>) =
    let names = stateNames dfa.States
    [ yield sprintf "states: %A" dfa.States
      yield sprintf "start state: %s" names[dfa.Start]
      yield "state encoding:"
      for KeyValue(state, name) in names do yield sprintf "%s = %A" name state
      yield "transitions:"
      for KeyValue((source, c), target) in dfa.Transitions do
          yield sprintf "delta(%s, %c) = %s" names[source] c names[target]
      yield sprintf "accepting states: %A" (dfa.Accepting |> Set.map (fun q -> names[q])) ]
    |> String.concat "\n"

let dfa2dot (dfa: Dfa<'state>) : string * Map<'state, string> =
    let names = stateNames dfa.States
    let lines =
        [ yield "digraph DFA {"
          yield "rankdir=LR;"
          yield "__start [shape=point, color=blue];"
          yield sprintf "__start -> %s;" names[dfa.Start]
          for state in dfa.States do
              let shape = if Set.contains state dfa.Accepting then "doublecircle" else "circle"
              yield sprintf "%s [shape=%s];" names[state] shape
          for KeyValue((source, c), target) in dfa.Transitions do
              yield sprintf "%s -> %s [label=%s];" names[source] names[target] (quote (string c))
          yield "}" ]
    String.concat "\n" lines, names

The function `nfa2string` converts a non-deterministic finite state machine `nfa` into a string.

In [ ]:
let nfa2string (nfa: Nfa<'state>) =
    [ yield sprintf "states: %A" nfa.States
      yield sprintf "start state: %A" nfa.Start
      yield "transitions:"
      for KeyValue((source, symbol), targets) in nfa.Transitions do
          let label = match symbol with Eps -> "𝜀" | Input c -> string c
          for target in targets do yield sprintf "[%A, %s] |-> %A" source label target
      yield sprintf "accepting states: %A" nfa.Accepting ]
    |> String.concat "\n"

let nfa2dot (nfa: Nfa<'state>) : string =
    let names = stateNames nfa.States
    [ yield "digraph NFA {"
      yield "rankdir=LR;"
      yield "__start [shape=point, color=blue];"
      yield sprintf "__start -> %s;" names[nfa.Start]
      for state in nfa.States do
          let shape = if Set.contains state nfa.Accepting then "doublecircle" else "circle"
          yield sprintf "%s [shape=%s, label=%s];" names[state] shape (quote (sprintf "%A" state))
      for KeyValue((source, symbol), targets) in nfa.Transitions do
          let label = match symbol with Eps -> "𝜀" | Input c -> string c
          for target in targets do
              yield sprintf "%s -> %s [label=%s];" names[source] names[target] (quote label)
      yield "}" ]
    |> String.concat "\n"

## Rendering
DOT generation uses only the .NET standard library. For optional SVG rendering, install Graphviz (`conda install -c conda-forge graphviz`, or the platform package manager). `renderSvg` invokes its `dot` executable with redirected input rather than a shell command. It returns SVG text that the notebook can display or save; an unavailable renderer gives a clear error.

In [ ]:
let renderSvg (dotSource: string) : string =
    let info = System.Diagnostics.ProcessStartInfo("dot")
    info.ArgumentList.Add("-Tsvg")
    info.UseShellExecute <- false
    info.RedirectStandardInput <- true
    info.RedirectStandardOutput <- true
    info.RedirectStandardError <- true
    use renderer = new System.Diagnostics.Process()
    renderer.StartInfo <- info
    try
        if not (renderer.Start()) then failwith "Graphviz did not start."
    with :? System.ComponentModel.Win32Exception ->
        failwith "Install Graphviz and make its dot executable available on PATH to render SVG."
    // Drain both streams concurrently; a renderer's stderr must not block its stdout.
    let output = renderer.StandardOutput.ReadToEndAsync()
    let errors = renderer.StandardError.ReadToEndAsync()
    renderer.StandardInput.Write(dotSource)
    renderer.StandardInput.Close()
    renderer.WaitForExit()
    let svg = output.GetAwaiter().GetResult()
    let diagnostics = errors.GetAwaiter().GetResult()
    if renderer.ExitCode <> 0 then failwith ("Graphviz rendering failed: " + diagnostics)
    svg

let saveDot (path: string) (source: string) = System.IO.File.WriteAllText(path, source)